In [1]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Imports from the grad-cam package
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

import sys
print(sys.executable)

/home/tony/venvs/depthAnythingV2/bin/python


In [4]:

# 1. Load a pretrained model (transfer learning is crucial for small datasets)
model = YOLO("yolo11n-seg.pt")

# 2. Train with optimizations for a small dataset
results = model.train(
    data='data.yaml',       
    epochs=200,             # Increased epochs, but we will rely on early stopping
    patience=30,            # Mentions early stopping: halts training if no improvement after 30 epochs
    batch=16,               # A stable, smaller batch size to keep gradients steady
    imgsz=640,              
    name='mango_seg_v1',
    
    # --- Small Dataset Optimizations ---
    freeze=10,              # Freezes the backbone layers. Keeps pretrained features intact and only trains the segmentation head.
    dropout=0.1,            # Adds regularization to prevent neurons from co-adapting/overfitting
    box=8.0,                # Slightly increase box loss gain if bounding boxes are loose
    cls=1.5,                # Slightly increase class loss gain to prioritize finding the mangoes
)

New https://pypi.org/project/ultralytics/8.4.77 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.40 🚀 Python-3.11.14 torch-2.11.0+cu130 CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=8.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=m

In [11]:
import os
import glob
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# ==========================================
# 1. CORE WRAPPERS AND TARGET CLASSES
# ==========================================
class YOLOModelWrapper(torch.nn.Module):
    """
    YOLOv11 returns a complex tuple during its forward pass. 
    This wrapper extracts only the raw prediction tensor needed by the CAM library.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        result = self.model(x)
        if isinstance(result, (tuple, list)):
            return result[0]  
        return result


class YOLOTarget:
    def __init__(self, category_id=0):
        self.category_id = category_id

    def __call__(self, model_output):
        preds = model_output
        score = torch.sum(preds[0, 4 + self.category_id, :])
        return score


# ==========================================
# 2. HELPER PIPELINE FUNCTION
# ==========================================
def generate_cams(img_path, wrapper_model, target_layers, targets, img_size=(640, 640)):
    """Preprocesses a single image, computes all 3 CAM variants, and returns them."""
    # 1. Read & Preprocess
    raw_img = cv2.imread(img_path)
    if raw_img is None:
        raise FileNotFoundError(f"Could not load image at {img_path}")
    
    raw_img_rgb = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(raw_img_rgb, img_size)
    img_float = np.float32(img_resized) / 255.0
    
    input_tensor = torch.from_numpy(img_float).transpose(0, 2).transpose(1, 2).unsqueeze(0)
    device = next(wrapper_model.parameters()).device
    input_tensor = input_tensor.to(device).requires_grad_(True)
    
    # 2. Instantiate Engines
    cam_methods = {
        "GradCAM": GradCAM(model=wrapper_model, target_layers=target_layers),
        "GradCAM++": GradCAMPlusPlus(model=wrapper_model, target_layers=target_layers),
        "EigenCAM": EigenCAM(model=wrapper_model, target_layers=target_layers) 
    }
    
    outputs = {}
    # 3. Generate Heatmaps
    with torch.enable_grad():
        for method_name, cam_engine in cam_methods.items():
            grayscale_cam = cam_engine(input_tensor=input_tensor, targets=targets)[0, :, :]
            cam_output = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)
            # Convert back to BGR for OpenCV rendering/saving
            outputs[method_name] = cv2.cvtColor(cam_output, cv2.COLOR_RGB2BGR)
            
    # Also return the original resized image in BGR format for standard comparisons
    outputs["Original"] = cv2.cvtColor(img_resized, cv2.COLOR_RGB2BGR)
    return outputs


# ==========================================
# 3. VISUALIZATION FUNCTIONS
# ==========================================
def show_single_image_cams(img_path, wrapper_model, target_layers, targets):
    """Generates CAMs for one image and displays them sequentially in interactive windows."""
    print(f"\nProcessing {img_path} for interactive display...")
    cams = generate_cams(img_path, wrapper_model, target_layers, targets)
    
    print("-> Press ANY KEY on the image window to cycle to the next heatmap. Press 'q' to exit early.")
    
    for name, img_data in cams.items():
        cv2.imshow(f"CAM Viewer - {name}", img_data)
        key = cv2.waitKey(0) & 0xFF
        cv2.destroyWindow(f"CAM Viewer - {name}")
        if key == ord('q'):
            break


def batch_process_folder(folder_path, wrapper_model, target_layers, targets, output_dir="./cam_comparisons"):
    """
    Processes all images in a folder and stacks them into a single side-by-side
    grid matrix (Original | GradCAM | GradCAM++ | EigenCAM) for easy benchmarking.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Match standard image extensions
    extensions = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
    image_paths = []
    for ext in extensions:
        image_paths.extend(glob.glob(os.path.join(folder_path, ext)))
        
    if not image_paths:
        print(f"No images found in folder: {folder_path}")
        return

    print(f"\nFound {len(image_paths)} images. Starting batch comparison process...")
    
    for idx, path in enumerate(image_paths):
        filename = os.path.basename(path)
        print(f"[{idx+1}/{len(image_paths)}] Processing: {filename}")
        
        try:
            cams = generate_cams(path, wrapper_model, target_layers, targets)
            
            # Combine horizontally: Original | GradCAM | GradCAM++ | EigenCAM
            comparison_grid = np.hstack([
                cams["Original"], 
                cams["GradCAM"], 
                cams["GradCAM++"], 
                cams["EigenCAM"]
            ])
            
            # Add text labels on top of each panel so you know exactly which is which
            h, w, _ = cams["Original"].shape
            labels = ["Original", "GradCAM", "GradCAM++", "EigenCAM"]
            for i, label in enumerate(labels):
                cv2.putText(comparison_grid, label, (10 + (i * w), 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
            
            # Save the diagnostic comparison sheet
            save_path = os.path.join(output_dir, f"compare_{filename}")
            cv2.imwrite(save_path, comparison_grid)
            
        except Exception as e:
            print(f"Skipping {filename} due to error: {e}")

    print(f"\nBatch processing finished! Check comparison grids in: '{output_dir}'")


# ==========================================
# 4. EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    # Setup model properties
    # yolo_model = YOLO("./runs/segment/train6/weights/best.pt")
    yolo_model = YOLO("./runs/segment/mango_seg_v1-4/weights/best.pt")

    pure_pytorch_model = YOLOModelWrapper(yolo_model.model)
    pure_pytorch_model.train()

    for param in pure_pytorch_model.parameters():
        param.requires_grad = True

    target_layers = [pure_pytorch_model.model.model[-2]]
    targets = [YOLOTarget(category_id=0)]  # Adjust your target class ID if Mango isn't 0
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pure_pytorch_model.to(device)

    # --- OPTION A: Show on screen via popup window ---
    # single_preview_img = "../../images/testSubjects/PXL_20260415_033937954.jpg"
    # show_single_image_cams(single_preview_img, pure_pytorch_model, target_layers, targets)
    
    # --- OPTION B: Batch process an entire directory into a side-by-side grid ---
    folder_to_test = "../../images/train/testSubjects/"
    batch_process_folder(folder_to_test, pure_pytorch_model, target_layers, targets)


Found 13 images. Starting batch comparison process...
[1/13] Processing: PXL_20260415_033647496.jpg
[2/13] Processing: PXL_20260415_033651760.jpg
[3/13] Processing: PXL_20260415_033715282.jpg
[4/13] Processing: PXL_20260415_033736357.jpg
[5/13] Processing: PXL_20260415_033739466.jpg
[6/13] Processing: PXL_20260415_033808357.jpg
[7/13] Processing: PXL_20260415_033816924.jpg
[8/13] Processing: PXL_20260415_033822573.jpg
[9/13] Processing: PXL_20260415_033834534.jpg
[10/13] Processing: PXL_20260415_033842071.jpg
[11/13] Processing: PXL_20260415_033850831.jpg
[12/13] Processing: PXL_20260415_033859036.MACRO_FOCUS.jpg
[13/13] Processing: PXL_20260415_033937954.jpg

Batch processing finished! Check comparison grids in: './cam_comparisons'
